# Module 2: the Document object

In [2]:
from langchain_core.documents import Document
doc = Document(
    # content
    page_content="Minecraft was created by Markus Persson, also known as Notch.",
    # where the content came from
    metadata={
        "source": "minecraft_history.txt",
        "chunk_id": 0
    }
)

print(type(doc))
print(doc.page_content)
print(doc.metadata)

<class 'langchain_core.documents.base.Document'>
Minecraft was created by Markus Persson, also known as Notch.
{'source': 'minecraft_history.txt', 'chunk_id': 0}


# Module 3: loading text files

In [3]:
from langchain_community.document_loaders import  DirectoryLoader, TextLoader

loader = DirectoryLoader(
    "mc_knowledge",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={
        "encoding": "utf-8"
    }
)

docs = loader.load()

print(type(docs))
print(len(docs))
print(type(docs[0]))
print(docs[0].page_content[:300])
print(docs[0].metadata)

/tmp/ipykernel_36873/2911601275.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import  DirectoryLoader, TextLoader
/home/razzannr/Documents/GitHub/minecraft_chatbot/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<class 'list'>
5
<class 'langchain_core.documents.base.Document'>
Minecraft is a sandbox game developed and published by the Swedish company Mojang Studios. Following its initial public alpha release as an early access title in 2009, it was formally released in November 2011 for personal computers. The game has since been ported to numerous platforms, including mo
{'source': 'mc_knowledge/minecraft_history.txt'}


# Module 4: splitting documents into chunks

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap=100
)

splits = text_splitter.split_documents(docs)

print(type(splits))
print(len(splits))
print(type(splits[0]))
print(splits[0].page_content[:300])
print(splits[0].metadata)

<class 'list'>
104
<class 'langchain_core.documents.base.Document'>
Minecraft is a sandbox game developed and published by the Swedish company Mojang Studios. Following its initial public alpha release as an early access title in 2009, it was formally released in November 2011 for personal computers. The game has since been ported to numerous platforms, including mo
{'source': 'mc_knowledge/minecraft_history.txt'}


# Module 5: embeddings with langchain + hugging face

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device":"cpu"}
)

vector = embeddings.embed_query("Who created Minecraft?")

print(type(vector))
print(type(vector[0]))
print(len(vector))
print(vector[:10])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12877.08it/s]


<class 'list'>
<class 'float'>
384
[-0.022937068715691566, 0.061923909932374954, -0.002757424023002386, -0.02537143975496292, -0.03191069886088371, -0.08562983572483063, 0.023514972999691963, -0.008422176353633404, -0.047711070626974106, 0.055575452744960785]


# Module 6: store with chroma

In [6]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory="./chroma_minecraft_db"
)

### testing the search database feature

In [ ]:
result = vectorstore.similarity_search(
    "Who created Minecraft?",
    k=3
)

print(type(result))
print(len(result))
print(type(result[0]))
print(result[0].page_content[:300])
print(result[0].metadata)

<class 'list'>
3
<class 'langchain_core.documents.base.Document'>
Minecraft is a sandbox game developed and published by the Swedish company Mojang Studios. Following its initial public alpha release as an early access title in 2009, it was formally released in November 2011 for personal computers. The game has since been ported to numerous platforms, including mo
{'source': 'mc_knowledge/minecraft_history.txt'}
page_content='Originally created by Markus "Notch" Persson using the Java programming language, Jens "Jeb" Bergensten was handed control over the game's development following its full release. In November 2014, Mojang and the Minecraft intellectual property were purchased by Microsoft for US$2.5 billion; Xbox Game Studios hold the publishing rights for the Bedrock Edition, the unified cross-platform version which evolved from the Pocket Edition codebase and replaced the legacy console versions. Bedrock is' metadata={'source': 'mc_knowledge/minecraft_history.txt'}


# Module 7: the retriever

In [9]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k":3}
)

retrieved_docs = retriever.invoke("Who created Minecraft?")

print(type(retrieved_docs))
print(len(retrieved_docs))
print(type(retrieved_docs[0]))

for doc in retrieved_docs:
    print(doc.page_content)
    print(doc.metadata)
    print()

<class 'list'>
3
<class 'langchain_core.documents.base.Document'>
Minecraft is a sandbox game developed and published by the Swedish company Mojang Studios. Following its initial public alpha release as an early access title in 2009, it was formally released in November 2011 for personal computers. The game has since been ported to numerous platforms, including mobile devices and various video game consoles.
{'source': 'mc_knowledge/minecraft_history.txt'}

Originally created by Markus "Notch" Persson using the Java programming language, Jens "Jeb" Bergensten was handed control over the game's development following its full release. In November 2014, Mojang and the Minecraft intellectual property were purchased by Microsoft for US$2.5 billion; Xbox Game Studios hold the publishing rights for the Bedrock Edition, the unified cross-platform version which evolved from the Pocket Edition codebase and replaced the legacy console versions. Bedrock is
{'source': 'mc_knowledge/minecraft_histor

# Module 8: formatting retrieved documents into context

In [ ]:
def format_docs(docs):
    return "\n\n".join(
        f"Source: {doc.metadata.get('source', 'unknown')}\nContent: {doc.page_content}"
        for doc in docs
    )

context = format_docs(retrieved_docs)
print(context)

Source: mc_knowledge/minecraft_history.txt
Content: Minecraft is a sandbox game developed and published by the Swedish company Mojang Studios. Following its initial public alpha release as an early access title in 2009, it was formally released in November 2011 for personal computers. The game has since been ported to numerous platforms, including mobile devices and various video game consoles.

Source: mc_knowledge/minecraft_history.txt
Content: Originally created by Markus "Notch" Persson using the Java programming language, Jens "Jeb" Bergensten was handed control over the game's development following its full release. In November 2014, Mojang and the Minecraft intellectual property were purchased by Microsoft for US$2.5 billion; Xbox Game Studios hold the publishing rights for the Bedrock Edition, the unified cross-platform version which evolved from the Pocket Edition codebase and replaced the legacy console versions. Bedrock is

Source: mc_knowledge/minecraft_gameplay.txt
Content

# Module 9: prompt template

In [12]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
    You are a Minecraft assistant.
    Answer the user's question using only the provided context.
    If the answer is not in the context, say: "I do not know based on the provided context."
    Do not invent facts.
    """
    ),
    (
        "user",
        """
    Context:
    {context}

    Question:
    {question}
    """
    )
])

In [13]:
formatted = prompt.invoke({
    "context": "Minecraft was created by Markus Persson, also known as Notch.",
    "question": "Who created Minecraft?"
})

print(type(formatted))
print(formatted)

<class 'langchain_core.prompt_values.ChatPromptValue'>
messages=[SystemMessage(content='\n    You are a Minecraft assistant.\n    Answer the user\'s question using only the provided context.\n    If the answer is not in the context, say: "I do not know based on the provided context."\n    Do not invent facts.\n    ', additional_kwargs={}, response_metadata={}), HumanMessage(content='\n    Context:\n    Minecraft was created by Markus Persson, also known as Notch.\n\n    Question:\n    Who created Minecraft?\n    ', additional_kwargs={}, response_metadata={})]


# Module 10: connect to qwen

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(model_name, dtype="auto")

hf_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
    do_sample=False,
    return_full_text=False
)

llm = HuggingFacePipeline(pipeline=hf_pipe)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 11002.90it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [7]:
result = llm.invoke("Explain Minecraft in one sentence.")
print(type(result))
print(result)

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


<class 'str'>
 Minecraft is a sandbox block-building game where players create and explore worlds filled with blocks, ores, and other resources using tools and crafting items to craft and upgrade their creations. It's an open-world adventure that encourages creativity, exploration, and teamwork among players. The game has become a global phenomenon due to its unique gameplay mechanics, diverse player base, and the ability to create immersive virtual environments. Players can build structures, farms, cities, and more, all while navigating through various dungeons and completing quests to unlock new areas and features. With over 10 million active users worldwide, Minecraft continues to be a popular platform for creative expression and community


# Module 11 and 12: the most basic RAG langchain for minecraft chatbot

In [8]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# =========================
# 1. Load documents
# =========================
loader = DirectoryLoader(
    "mc_knowledge",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={
        "encoding": "utf-8"
    }
)

docs = loader.load()

# =========================
# 2. Split documents
# =========================
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap=100
)

splits = text_splitter.split_documents(docs)

# =========================
# 3. Embeddings
# =========================
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device":"cpu"}
)

vector = embeddings.embed_query("Who created Minecraft?")

# =========================
# 4. Vector store
# =========================
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory="./chroma_minecraft_db"
)

# =========================
# 5. Retriever
# =========================
retriever = vectorstore.as_retriever(
    search_kwargs={"k":3}
)

# =========================
# 6. Load Qwen
# =========================
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(model_name, dtype="auto")

# =========================
# 7. Format docs
# =========================
def format_docs(docs):
    return "\n\n".join(
        f"Source: {doc.metadata.get('source', 'unknown')}\nContent: {doc.page_content}"
        for doc in docs
    )

# =========================
# 8. Manual Qwen generator
# =========================

def generate_with_qwen(question, context, tokenizer, model):
    messages = [
        {
            "role": "system",
            "content": """
You are a Minecraft assistant.
Answer the user's question using only the provided context.
If the answer is not in the context, say: "I do not know based on the provided context."
Do not invent facts.
"""
        },
        {
            "role": "user",
            "content": f"""
Context:
{context}

Question:
{question}
"""
        }
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer(
        prompt_text,
        return_tensors="pt"
    ).to(model.device)

    output_ids = model.generate(
        **model_inputs,
        max_new_tokens=128,
        do_sample=False
    )

    input_length = model_inputs["input_ids"].shape[1]
    new_output_ids = output_ids[:, input_length:]

    answer = tokenizer.decode(
        new_output_ids[0],
        skip_special_tokens=True
    )

    return answer

# =========================
# 9. Ask function
# =========================
def ask_langchain_rag(question):
    retrieved_docs = retriever.invoke(question)

    context = format_docs(retrieved_docs)

    answer = generate_with_qwen(
        question=question,
        context=context,
        tokenizer=tokenizer,
        model=model
    )

    return answer, retrieved_docs

# =========================
# 10. Chat loop
# =========================
while True:
    question = input("User: ")

    if question.lower().strip() in ["exit", "quit", "q"]:
        print("Bot: Goodbye!")
        break

    answer, retrieved_docs = ask_langchain_rag(question)

    print("\nBot:", answer)

    print("\nRetrieved documents:")
    for doc in retrieved_docs:
        print("-", doc.metadata.get("source", "unknown"))
        print(doc.page_content[:200])
        print()

    print()

/tmp/ipykernel_26001/2828545580.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 11010.96it/s]



Bot: Minecraft is a sandbox video game that allows players to create their own worlds and explore them freely. It is available on multiple platforms, including PC, console, and mobile devices. Players can choose from different gameplay modes, such as first-person or third-person, and customize their character's appearance and abilities through achievements. The game features a variety of environments, including mountains, forests, and oceans, and includes optional achievements like completing certain tasks or achieving specific milestones.

Retrieved documents:
- mc_knowledge/minecraft_history.txt
Minecraft is a sandbox game developed and published by the Swedish company Mojang Studios. Following its initial public alpha release as an early access title in 2009, it was formally released in Nove

- mc_knowledge/minecraft_history.txt
Minecraft is a sandbox game developed and published by the Swedish company Mojang Studios. Following its initial public alpha release as an early access ti